In [4]:
import sys
sys.path.append('..')
import module

In [5]:
import os
from pathlib import Path


BASE_DATASOURCE_PATH=os.path.join(Path.home().as_posix(), "datasets", "anxiety_raw")
BASE_ANNOTATION_PATH=os.path.join(BASE_DATASOURCE_PATH, "annotations.xlsx")

assert os.path.exists(BASE_DATASOURCE_PATH), f"Path tidak ditemukan: {BASE_DATASOURCE_PATH}"
assert os.path.exists(BASE_ANNOTATION_PATH), f"Path tidak ditemukan: {BASE_ANNOTATION_PATH}"

print("Datasource path successfully set.")

Datasource path successfully set.


In [6]:
import re
import pandas as pd
import numpy as np


dfs = pd.read_excel(BASE_ANNOTATION_PATH, sheet_name=['before', 'after'], engine='openpyxl')

df_before = dfs['before']

# Ekstraksi nomor q dari filepath menggunakan regex
def extract_q_name(path: str):
    m = re.search(r'/q(\d+)(?:/|$)', path)
    return int(m.group(1)) if m else None

# Menambahkan kolom q_num berdasarkan ekstraksi dari filepath
df = df_before.copy()
df['q_num'] = df['filepath'].apply(extract_q_name)

# Mengurutkan data berdasarkan q_num dan subject_name
df_sorted_by_q = df.sort_values(by='q_num', na_position='last')
df_sorted = df.sort_values(by=['subject_name', 'q_num'], na_position='last')

df_sorted.head()

,subject_name,anxiety_level,filepath,q_num
33,aaisyah_nursalsabiil_ni_patriarti,high,/home/inadio/datasets/anxiety_raw/before/anxie...,1
34,aaisyah_nursalsabiil_ni_patriarti,high,/home/inadio/datasets/anxiety_raw/before/anxie...,2
30,aaisyah_nursalsabiil_ni_patriarti,high,/home/inadio/datasets/anxiety_raw/before/anxie...,3
31,aaisyah_nursalsabiil_ni_patriarti,high,/home/inadio/datasets/anxiety_raw/before/anxie...,4
32,aaisyah_nursalsabiil_ni_patriarti,high,/home/inadio/datasets/anxiety_raw/before/anxie...,5


In [7]:
import os
import gc
import numpy as np
from pathlib import Path
from src.apex.modules.v2 import ApexPhaseSpotter, ApexPhaseVisualizer

spotter = ApexPhaseSpotter()
magnitudes = []

for index, row in df_sorted.iterrows():
    subject_name = row["subject_name"]
    anxiety_level = row["anxiety_level"]
    video_path = os.path.join(BASE_DATASOURCE_PATH, row["filepath"])
    question_number = Path(video_path).parents[0].name

    save_path = f"../.output/{subject_name}_{question_number}_{anxiety_level}.npy"
    
    if os.path.exists(save_path):
        print(f"File {save_path} sudah ada, skip.")
        continue

    print(f"Processing {video_path}...")
    spotter.process(video_path)

    flow_array = np.array([
        spotter.horizontal_magnitudes,
        spotter.vertical_magnitudes,
        spotter.magnitudes
    ])

    data = np.array([subject_name,
                     question_number,
                     anxiety_level,
                     flow_array], dtype=object)

    np.save(save_path, data)

    del flow_array
    del data
    gc.collect()


W0000 00:00:1772933640.311076   69331 face_landmarker_graph.cc:174] Sets FaceBlendshapesGraph acceleration to xnnpack by default.
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1772933640.316972   69332 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1772933640.326727   69332 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


File ../.output/aaisyah_nursalsabiil_ni_patriarti_q1_high.npy sudah ada, skip.
File ../.output/aaisyah_nursalsabiil_ni_patriarti_q2_high.npy sudah ada, skip.
File ../.output/aaisyah_nursalsabiil_ni_patriarti_q3_high.npy sudah ada, skip.
File ../.output/aaisyah_nursalsabiil_ni_patriarti_q4_high.npy sudah ada, skip.
File ../.output/aaisyah_nursalsabiil_ni_patriarti_q5_high.npy sudah ada, skip.
File ../.output/abdillah_agil_arbiansyah_q1_low.npy sudah ada, skip.
File ../.output/abdillah_agil_arbiansyah_q2_low.npy sudah ada, skip.
File ../.output/abdillah_agil_arbiansyah_q3_low.npy sudah ada, skip.
File ../.output/abdillah_agil_arbiansyah_q4_low.npy sudah ada, skip.
File ../.output/abdillah_agil_arbiansyah_q5_low.npy sudah ada, skip.
File ../.output/abdul_aziz_q1_low.npy sudah ada, skip.
File ../.output/abdul_aziz_q2_low.npy sudah ada, skip.
File ../.output/abdul_aziz_q3_low.npy sudah ada, skip.
File ../.output/abdul_aziz_q4_low.npy sudah ada, skip.
File ../.output/abdul_aziz_q5_low.npy su